# Notebook 05: Conversation Memory

**Goal:** Implement sliding window conversation memory ($k=5$) for multi-turn Q&A, context-aware follow-up handling, turn eviction, and memory reset.

## Objectives:
1. Initialize `ConversationBufferWindowMemory` with $k=5$
2. Test multi-turn dialogue with follow-up resolution
3. Test sliding window eviction after exceeding 5 turns
4. Test explicit memory clearing and state reset
5. Run verification assertions for memory integrity

In [1]:
# Cell 1: Import dependencies & sliding window memory wrapper
import os
import sys
from typing import List, Dict, Tuple

project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

class SlidingWindowMemory:
    """Sliding window conversation memory retaining last k turns."""
    def __init__(self, k: int = 5):
        self.k = k
        self.history: List[Tuple[str, str]] = []  # [(user_query, ai_response)]
        
    def add_turn(self, user_query: str, ai_response: str):
        self.history.append((user_query, ai_response))
        if len(self.history) > self.k:
            self.history.pop(0)  # Evict oldest turn
            
    def get_formatted_history(self) -> str:
        if not self.history:
            return "None"
        lines = []
        for u, a in self.history:
            lines.append(f"User: {u}\nAssistant: {a}")
        return "\n".join(lines)
        
    def clear(self):
        self.history.clear()

memory = SlidingWindowMemory(k=5)
print("SlidingWindowMemory (k=5) initialized successfully.")

SlidingWindowMemory (k=5) initialized successfully.


In [2]:
# Cell 2: Multi-Turn Interaction Test
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import dotenv

dotenv.load_dotenv(os.path.join(project_root, ".env"))
llm = ChatOllama(model="llama3", base_url=os.getenv("OLLAMA_HOST", "http://localhost:11434"), temperature=0.0)

MEMORY_PROMPT_TEMPLATE = """You are a document Q&A assistant.
Conversation History:
{history}

Current Context:
{context}

Current Question: {question}
Answer:
"""

memory_prompt = ChatPromptTemplate.from_template(MEMORY_PROMPT_TEMPLATE)
chain = memory_prompt | llm | StrOutputParser()

context = """
ConverseIQ uses BM25 sparse retrieval combined with CrossEncoder reranking (ms-marco-MiniLM-L-6-v2).
BM25 latency is under 15ms and reranking latency is 150-300ms on CPU.
"""

# Turn 1
q1 = "What reranker model does ConverseIQ use?"
a1 = chain.invoke({"history": memory.get_formatted_history(), "context": context, "question": q1})
memory.add_turn(q1, a1.strip())
print(f"Turn 1 Q: '{q1}'")
print(f"Turn 1 A: {a1.strip()}\n")

# Turn 2 (Follow-up referencing Turn 1)
q2 = "What is its average latency on CPU?"
a2 = chain.invoke({"history": memory.get_formatted_history(), "context": context, "question": q2})
memory.add_turn(q2, a2.strip())
print(f"Turn 2 Q: '{q2}' (Follow-up)")
print(f"Turn 2 A: {a2.strip()}")

Turn 1 Q: 'What reranker model does ConverseIQ use?'
Turn 1 A: ConverseIQ uses the ms-marco-MiniLM-L-6-v2 CrossEncoder as its reranker model.



Turn 2 Q: 'What is its average latency on CPU?' (Follow-up)
Turn 2 A: According to the current context, BM25 latency is under 15ms and reranking latency (using ms-marco-MiniLM-L-6-v2) is 150-300ms. Therefore, the average latency on CPU would be a combination of these two values.


In [3]:
# Cell 3: Sliding Window Eviction Test (k=5)
print("Adding turns 3 to 7 to test sliding window eviction...")
for i in range(3, 8):
    u_q = f"Question number {i}?"
    u_a = f"Answer number {i}."
    memory.add_turn(u_q, u_a)

print(f"Total turns recorded in memory buffer: {len(memory.history)} (Max capacity k=5)")
print("Current Memory Contents:\n", memory.get_formatted_history())

Adding turns 3 to 7 to test sliding window eviction...
Total turns recorded in memory buffer: 5 (Max capacity k=5)
Current Memory Contents:
 User: Question number 3?
Assistant: Answer number 3.
User: Question number 4?
Assistant: Answer number 4.
User: Question number 5?
Assistant: Answer number 5.
User: Question number 6?
Assistant: Answer number 6.
User: Question number 7?
Assistant: Answer number 7.


In [4]:
# Cell 4: Memory Reset Test
memory.clear()
print(f"Memory after reset: '{memory.get_formatted_history()}' (Count: {len(memory.history)})")

Memory after reset: 'None' (Count: 0)


In [5]:
# Cell 5: Verification Assertions for Notebook 05
print("Running verification assertions on Conversation Memory...")
# 1. Follow-up answer should incorporate context
assert len(a2.strip()) > 0, "Verification Failed: Follow-up answer is empty!"

# 2. Sliding window memory limit k=5 enforced
assert len(memory.history) == 0, "Verification Failed: Memory was not reset!"

test_mem = SlidingWindowMemory(k=5)
for i in range(1, 8):
    test_mem.add_turn(f"Q{i}", f"A{i}")
assert len(test_mem.history) == 5, f"Verification Failed: Expected 5 items in memory, found {len(test_mem.history)}"
assert test_mem.history[0][0] == "Q3", f"Verification Failed: Expected Q3 to be oldest turn, found {test_mem.history[0][0]}"

print("✅ ALL VERIFICATION TESTS PASSED FOR NOTEBOOK 05!")

Running verification assertions on Conversation Memory...
✅ ALL VERIFICATION TESTS PASSED FOR NOTEBOOK 05!
